# TerraMind: fine-tune the intent and NER models on a Colab GPU

Before running: **Runtime > Change runtime type > T4 GPU**.

This notebook runs the same scripts you run locally (`scripts/train_intent.py` and `scripts/train_ner.py` with `--model full`). It only supplies the GPU.

## 1. Get the project

Option A: zip the `TerraMind` folder on your laptop **without** `.venv`, `models` and `.git`, then upload it with the cell below.

Option B: if the project is on GitHub, replace the cell with `!git clone <your repo url> TerraMind`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose TerraMind.zip
!unzip -q -o TerraMind.zip -d .

## 2. Install dependencies and check the GPU

Colab already has a GPU build of PyTorch, so `torch` is skipped.

In [ ]:
%cd TerraMind
!grep -v '^torch' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
import torch
print('GPU available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU: change the runtime type')

## 3. Check the code works here

In [ ]:
!python -m pytest -q tests/nlp/test_intent.py tests/nlp/test_ner.py tests/nlp/test_dataset.py

## 4. Train `xlm-roberta-base` with three seeds each

Each script prints its per-language summary at the end.

In [ ]:
!python scripts/train_intent.py --model full
!python scripts/train_ner.py --model full

## 5. Download the results

Unzip the download into your local `TerraMind/models/` folder so that `models/intent/full/summary.json` and `models/ner/full/summary.json` exist. Then set `active_run: full` under both `intent` and `ner` in `configs/nlp.yaml`.

Only each task's best seed keeps its weights, to keep the download near 2 GB. Every seed keeps its `metrics.json`.

In [ ]:
import json, shutil, pathlib
for task in ('intent', 'ner'):
    run = pathlib.Path('models') / task / 'full'
    best = json.loads((run / 'summary.json').read_text())['best_seed']
    for seed_dir in run.glob('seed_*'):
        if seed_dir.name != f'seed_{best}':
            (seed_dir / 'model.safetensors').unlink(missing_ok=True)
shutil.make_archive('terramind_models_full', 'zip', 'models', '.')
files.download('terramind_models_full.zip')